# 02 — Shapes, broadcasting, and matrix multiplication

**Level:** Beginner · **Study time:** 45–60 minutes (including experiments)

**Prerequisites:** Notebook 01.

## What you will be able to do
- Trace batch and feature axes.
- Distinguish elementwise operations from matrix multiplication.
- Prevent silent broadcasting errors.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 01](01_tensors.ipynb) · [Next: 03](03_autograd.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Write a layer as matrix arithmetic

A batch `X` with shape `[B, F]` multiplied by weights `[F, H]` produces `[B, H]`. Adding a bias `[H]` repeats that bias across the batch without manually copying it.


In [ ]:
X = torch.randn(5, 3)
W = torch.randn(3, 2)
b = torch.tensor([1., -1.])
Y = X @ W + b
print("X:", X.shape, "W:", W.shape, "Y:", Y.shape)
assert Y.shape == (5, 2)
assert torch.allclose(Y[0], X[0] @ W + b)


## Step 3 — See the difference between multiply and matmul

`*` multiplies corresponding elements, while `@` sums products along a shared axis. Broadcasting aligns dimensions from the right; dimensions must match or one must be 1.


In [ ]:
A = torch.tensor([[1., 2.], [3., 4.]])
print("Elementwise:", A * A)
print("Matrix multiply:", A @ A)
row = torch.tensor([[10., 20., 30.]])
column = torch.tensor([[1.], [2.]])
print("[1,3] + [2,1] -> [2,3]:", row + column)


## Step 4 — Catch a silent loss bug

Subtracting predictions `[B,1]` and targets `[B]` creates `[B,B]`, not one error per example. Code can run and still solve the wrong problem. Check shapes before computing the loss.


In [ ]:
pred = torch.tensor([[1.], [2.], [3.]])
target = torch.tensor([1., 2., 3.])
wrong_error = pred - target
right_error = pred - target.unsqueeze(1)
print("Wrong shape:", wrong_error.shape, "Correct shape:", right_error.shape)
assert right_error.shape == pred.shape
assert right_error.square().mean() == 0
assert wrong_error.square().mean() > 0


## Step 5 — Reorder image axes

PyTorch convolution layers expect `[batch, channels, height, width]`. `permute` reorders axes; `reshape` changes grouping. Reshaping cannot replace permuting because it does not express the same reordering.


In [ ]:
images_hwc = torch.arange(2 * 4 * 4 * 3).reshape(2, 4, 4, 3)
images_chw = images_hwc.permute(0, 3, 1, 2)
flat = images_chw.reshape(2, -1)
print("Image:", images_chw.shape, "Flat:", flat.shape)
assert images_chw[0, 2, 1, 3] == images_hwc[0, 1, 3, 2]


## Your turn — practice before opening the solutions

1. Standardize a `[10, 3]` batch separately for each feature. Hint: reduce over dim 0 with `keepdim=True`.
2. Add a batch axis to a `[3, 8, 8]` image.
3. Create batched matrices `[4, 2, 3]` and `[4, 3, 5]`; predict and verify the output shape.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
batch = torch.randn(10, 3)
z = (batch - batch.mean(0, keepdim=True)) / batch.std(0, keepdim=True).clamp_min(1e-6)
assert torch.allclose(z.mean(0), torch.zeros(3), atol=1e-6)
assert torch.randn(3, 8, 8).unsqueeze(0).shape == (1, 3, 8, 8)
assert (torch.randn(4, 2, 3) @ torch.randn(4, 3, 5)).shape == (4, 2, 5)


## Before moving on

Before executing a model, write the shape after every operation on paper. A successful operation is not proof that the shape is semantically correct.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)


[Course guide](../README.md) · [Previous: 01](01_tensors.ipynb) · [Next: 03](03_autograd.ipynb)
